# Marker Based Validation DLPFC

In [ ]:
import pandas as pd
import scanpy as sc
import anndata as ad
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plt


## 1. Load and prepare data for marker-based validation 

In [ ]:
#### Load scANVI annotate query file

adata_query_X = ad.read_h5ad("/tscc/lustre/ddn/scratch/aopatel/adata_scANVI_DLPFC.h5ad")

In [ ]:
#### Remember leiden clustering was performed on query+ref (full)

adata_query_X.obs = adata_query_X.obs.rename(
    columns={"leiden": "full_leiden"}
)

In [ ]:
adata_query_X = adata_query_X[adata_query_X.obs['cluster_flagged'] == 'N'].copy()
adata_query_X 

In [ ]:
adata_query_X.obs['is_high_confidence_95'] = adata_query_X.obs['sub_class_prob_max'] > 0.95
adata_query_X.obs['is_high_confidence_90'] = adata_query_X.obs['sub_class_prob_max'] > 0.90
adata_query_X.obs['is_high_confidence_85'] = adata_query_X.obs['sub_class_prob_max'] > 0.85

In [ ]:
#### Keep high confidence cells 

adata_query_X = adata_query_X[adata_query_X.obs['is_high_confidence_95']].copy()

In [ ]:
#### Get cell types with >= 100 cells

cell_type_counts = adata_query_X.obs['predictions'].value_counts()
keep_cell_types = cell_type_counts[cell_type_counts >= 100].index.tolist()

adata_query_X = adata_query_X[adata_query_X.obs['predictions'].isin(keep_cell_types)].copy()


In [ ]:
sc.pp.neighbors(adata_query_X, use_rep="X_scANVI")
sc.tl.umap(adata_query_X)

In [ ]:
adata_query_X

In [ ]:
sc.pl.umap(
    adata_query_X,
    color="predictions",
    frameon=False,
    legend_loc="on data"
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color="full_leiden",
    frameon=False,
)

In [ ]:
#### Keep raw data for DEG analysis and other analytics

adata_query_X.layers['counts']=adata_query_X.X.copy()

In [ ]:
#### Normalize and transform data 
#### (Ahlmann-Eltz & Huber, 2023, Nature Methods)

sc.pp.normalize_total(adata_query_X)
sc.pp.log1p(adata_query_X)



<div class="alert alert-block alert-info">
<b> After saving UMAP and normalizing counts start here!

</div>

In [ ]:
## adata_query_X.write_h5ad("/tscc/lustre/ddn/scratch/aopatel/adata_DLPFC_Finalized.h5ad")
adata_query_X =ad.read_h5ad("/tscc/lustre/ddn/scratch/aopatel/adata_DLPFC_Finalized.h5ad")

In [ ]:
sc.pl.umap(
    adata_query_X,
    color="predictions",
    frameon=False,
    legend_loc="on data",
    legend_fontsize=6,
    legend_fontoutline=2
)

In [ ]:
adata_query_X

In [ ]:
parent_map = {
    # Excitatory Neurons (Glutamatergic)
    "EN_L2_3_IT": "Excitatory",
    "EN_L3_5_IT_1": "Excitatory", "EN_L3_5_IT_2": "Excitatory", "EN_L3_5_IT_3": "Excitatory",
    "EN_L5_ET": "Excitatory", "EN_L5_6_NP": "Excitatory",
    "EN_L6_CT": "Excitatory", "EN_L6_IT_1": "Excitatory", "EN_L6_IT_2": "Excitatory",
    "EN_L6B": "Excitatory",

    # Inhibitory Neurons (GABAergic)
    "IN_PVALB": "Inhibitory", "IN_PVALB_CHC": "Inhibitory",
    "IN_SST": "Inhibitory", "IN_VIP": "Inhibitory",
    "IN_ADARB2": "Inhibitory",
    "IN_LAMP5_RELN": "Inhibitory", "IN_LAMP5_LHX6": "Inhibitory",

    # Glia
    "Astro": "Glia", "OPC": "Glia", "Oligo": "Glia",
    "Micro": "Glia", "PVM": "Glia",

    # Vascular
    "Endo": "Vascular", "VLMC": "Vascular", "PC": "Vascular",
}

# Create the new 'parent' column
adata_query_X.obs['parent'] = adata_query_X.obs['predictions'].map(parent_map)

# Verify no unmapped cell types
print(f"Unmapped cells: {adata_query_X.obs['parent'].isna().sum()}")
print(adata_query_X.obs['parent'].value_counts())

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=['full_leiden'],
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color="full_leiden",
    legend_loc="on data",   # puts numbers on clusters
    legend_fontsize=10,
    legend_fontoutline=2
)

In [ ]:
print(adata_query_X.obs["predictions"].value_counts())

## 2. Glial Verification 

### A. Preliminary Scoring Scheme

In [ ]:
#### Create preliminary scoring scheme

broad_neuronal_markers=['RBFOX3', 'ELAVL4','SNCA', 'MAP2','SNAP25', 'TUBB3', 'SYT1'] 

astrocyte_markers=['GFAP', 'AQP4', 'ALDH1L1', 'SLC1A2','SLC14A1','ACSBG1']

oligodendrocyte_markers=['MBP', 'MOG', 'PLP1', 'CLDN11', 'MYRF']

opc_markers=['PDGFRA', 'CSPG4', 'VCAN', 'PCDH15', 'LRP1']

microglia_markers=['ITGAM', 'P2RY12', 'CX3CR1','CSF1R','SLC2A5','GPR34']

glial_markers = (
    astrocyte_markers
    + oligodendrocyte_markers
    + opc_markers
    + microglia_markers
)




In [ ]:
#### Calculate neuronal_score
sc.tl.score_genes(adata_query_X, broad_neuronal_markers, score_name='neuronal_score')

#### Calculate non-neuronal_score
sc.tl.score_genes(adata_query_X, glial_markers, score_name='glial_score')

#### Calculate astrocyte
sc.tl.score_genes(adata_query_X, astrocyte_markers, score_name='astrocyte_score')

#### Calculate Oligodendrocyte score 
sc.tl.score_genes(adata_query_X, opc_markers, score_name='opc_score')

#### Calculate OPC score 
sc.tl.score_genes(adata_query_X, oligodendrocyte_markers, score_name='oligo_score')

#### Calculate Microglia score 
sc.tl.score_genes(adata_query_X, microglia_markers, score_name='micro_score')

### B. Neuron vs Glia

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=['neuronal_score','glial_score'],
    frameon=False,
)

### C. Glia

#### 1. Astrocyte

In [ ]:
sc.pl.umap(
    adata_query_X,
    color="astrocyte_score",
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=astrocyte_markers,
    frameon=False
)

### SLC1A3 does have expresion in PVMs

#### 2. Oligodendrocytes

In [ ]:
sc.pl.umap(
    adata_query_X,
    color="oligo_score",
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=oligodendrocyte_markers,
    frameon=False
)

#### 3. Oligodendrocyte Precursor Cells


In [ ]:
sc.pl.umap(
    adata_query_X,
    color="opc_score",
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=opc_markers,
    frameon=False
)

#### 4. Microglia/PVM

In [ ]:
sc.pl.umap(
    adata_query_X,
    color="micro_score",
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=microglia_markers,
    frameon=False,
)

## 3. Neurons

### A. Preliminary Scoring Scheme

In [ ]:
#### Two Classes of Neurons
inhib_markers=['GAD1','GAD2','ADARB2','SLC32A1']

excit_markers=['SLC17A7','NEUROD2','SATB2','TBR1','RORB','NRGN','CAMK2A']


In [ ]:
#### Calculate Inhibitory score 
sc.tl.score_genes(adata_query_X, inhib_markers, score_name='inhib_score')

#### Calculate Excitatory score
sc.tl.score_genes(adata_query_X, excit_markers, score_name='excit_score')

#### VIP markers
sc.tl.score_genes(adata_query_X, excit_markers, score_name='vip_score')


### B. Inhibitory vs Excitatory Neurons

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=['inhib_score','excit_score'],
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=inhib_markers,
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=excit_markers,
    frameon=False,
)

### C. Inhibitory Neurons

#### 1. VIP

In [ ]:
vip_markers=['VIP','CALB2','CHRNA2','SCTR','NPR3','GALNTL6', 'LRP1B',
            'GRM7', 'KCNT2', 'THSD7A', 'ERBB4', 'SYNPR', 'ADARB2', 'SLC24A3']

#### VIP markers
sc.tl.score_genes(adata_query_X, vip_markers, score_name='vip_score')

In [ ]:
sc.pl.umap(
    adata_query_X,
    color='vip_score',
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=vip_markers,
    frameon=False,
)

#### 2. SNCG/ADARB2

In [ ]:
adarb2_markers=['CXCL14','CNR1','CHRNA7', 'SLC8A1', 'ASIC2', 'MAML3', 'ADARB2', 'NPAS3', 'CNTN5', 'FSTL5','SNCG']
 

sc.tl.score_genes(adata_query_X, adarb2_markers, score_name='adarb2_score')

In [ ]:
sc.pl.umap(
    adata_query_X,
    color='adarb2_score',
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=adarb2_markers,
    frameon=False,
)

#### 3. PVALB

In [ ]:
pvalb_markers=['PVALB','LHX6','KCNS3','TAC1','SULF1', 'MYO5B', 
              'ADAMTS17', 'ERBB4', 'DPP10', 'ZNF804A', 'MYO16', 'GRIA4', 'SLIT2', 'SDK1']


#### pvalb markers
sc.tl.score_genes(adata_query_X, pvalb_markers, score_name='pvalb_score')

In [ ]:
sc.pl.umap(
    adata_query_X,
    color='pvalb_score',
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=pvalb_markers,
    frameon=False,
)

#### 4. SST

In [ ]:
sst_markers=['SST','ELFN1', 'CALB1','LHX6', 'GRIN3A', 'GRIK1', 'NMU',
             'RALYL', 'TRHDE', 'GRID2', 'NXPH1', 'COL25A1', 'SLC8A1', 'SOX6', 'ST6GALNAC5']



#### SST markers
sc.tl.score_genes(adata_query_X, sst_markers, score_name='sst_score')

In [ ]:
sc.pl.umap(
    adata_query_X,
    color='sst_score',
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=sst_markers,
    frameon=False,
)

#### 5. PVALB CHC (Chandelier)

In [ ]:
chand_markers=['PVALB','UNC5B','PTHLH','COL15A1','PLEKHH2','HTR1F','NPNT', 'COL21A1', 'SRGAP1']

sc.tl.score_genes(adata_query_X, chand_markers, score_name='chand_score')

In [ ]:
sc.pl.umap(
    adata_query_X,
    color='chand_score',
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=chand_markers,
    frameon=False,
)

#### 6. LAMP5

In [ ]:
lamp5_markers=['LAMP5','KIT','FGF13','SV2C','CPLX3',
              'PTPRT', 'PRELID2', 'GRIA4', 'RELN', 'PTCHD4', 'EYA4', 'MYO16', 'FBXL7']

#### SST markers
sc.tl.score_genes(adata_query_X, lamp5_markers, score_name='lamp5_score')

In [ ]:
sc.pl.umap(
    adata_query_X,
    color='lamp5_score',
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=lamp5_markers,
    frameon=False,
)

### D. Excitatory Neurons

#### 1. L2/3 IT

In [ ]:
l23it_markers=['CUX2','LINC00507','FREM3','PDGFD','COL5A2','CBLN2']


sc.tl.score_genes(adata_query_X, l23it_markers, score_name='l23it_score')

In [ ]:
sc.pl.umap(
    adata_query_X,
    color='l23it_score',
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=l23it_markers,
    frameon=False,
)

#### 2. L3-L5 IT

In [ ]:
l35it_markers = [
    'RORB',      # canonical L4/5 IT marker — strongest
    'THEMIS',    # L5 IT marker
    'CPNE4',     # L5 IT
    'CHN2',      # L5 IT
    'RXFP1',     # L5 IT specific
    'LINC01202', # L4 IT DLPFC enriched
    'CUX2',      # upper layer — shared with L2/3 but present
]


sc.tl.score_genes(adata_query_X, l5it_markers, score_name='l35it_score')

In [ ]:
sc.pl.umap(
    adata_query_X,
    color='l35it_score',
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=l35it_markers,
    frameon=False,
)

#### 3. L5/6 NP

In [ ]:
l56NP_markers=['TSHZ2', 'HTR2C', 'ITGA8', 'ZNF385D', 'ASIC2', 'CDH6', 'CRYM', 'NXPH2', 'CPNE4']


sc.tl.score_genes(adata_query_X, l56NP_markers, score_name='l56NP_score')

In [ ]:
sc.pl.umap(
    adata_query_X,
    color='l56NP_score',
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=l56NP_markers,
    frameon=False,
)

#### 4. L6 IT

In [ ]:
l6it_markers=['PDZRN4', 'CDH9', 'THEMIS', 'CDH13', 'MLIP', 'NFIA','LINC00343',]


sc.tl.score_genes(adata_query_X, l6it_markers, score_name='l6it_score')

In [ ]:
sc.pl.umap(
    adata_query_X,
    color='l6it_score',
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=l6it_markers,
    frameon=False,
)

#### 5. L6 IT Car3

In [ ]:
l6itcar3_markers=['THEMIS', 'RNF152', 'NTNG2', 'STK32B', 'KCNMB2', 'GAS2L3', 
                  'OLFML2B', 'POSTN', 'B3GAT2', 'NR4A2']


sc.tl.score_genes(adata_query_X, l6itcar3_markers, score_name='l6itcar3_score')

In [ ]:
sc.pl.umap(
    adata_query_X,
    color='l6itcar3_score',
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=l6itcar3_markers,
    frameon=False,
)

#### 6. L6b

In [ ]:
l6b_markers=['HS3ST4', 'MDFIC', 'CDH9', 'TLE4', 'FOXP2', 
             'KIAA1217','PCOLCE2','NPFFR2', 'NTM']


sc.tl.score_genes(adata_query_X, l6b_markers, score_name='l6b_score')

In [ ]:
sc.pl.umap(
    adata_query_X,
    color='l6b_score',
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=l6b_markers,
    frameon=False,
)

#### 6. L6 CT

In [ ]:
l6ct_markers=['ADAMTSL1', 'SORCS1', 'HS3ST4', 'TRPM3', 
              'TOX', 'SEMA3E', 'MEIS2', 'SEMA5A']


sc.tl.score_genes(adata_query_X, l6ct_markers, score_name='l6ct_score')

In [ ]:
sc.pl.umap(
    adata_query_X,
    color='l6ct_score',
    frameon=False,
)

In [ ]:
sc.pl.umap(
    adata_query_X,
    color=l6ct_markers,
    frameon=False,
)

In [ ]:
sc.tl.rank_genes_groups(adata_query_X,groupby="predictions")

In [ ]:
df=sc.get.rank_genes_groups_df(adata_query_X, group='L5 IT')

In [ ]:
# Filter for genes with high fold change and significance
top_markers = df[(df['pvals_adj'] < 0.05) & (df['logfoldchanges'] > 1.5)]

# Sort by the highest logfoldchange
top_markers = top_markers.sort_values('logfoldchanges', ascending=False)

print(top_markers.head(50))

In [ ]:
## Inhibitory vs Inhibitory

In [ ]:
# 1. Create  subset
adata_sub = adata_query_X[adata_query_X.obs['predictions'] == 'Lamp5'].copy()

# 2. Identify which 'full_leiden' groups have at least 2 cells in this subset
counts = adata_sub.obs['full_leiden'].value_counts()
keep_groups = counts[counts > 1].index

# 3. Filter the subset to only include those groups
adata_sub = adata_sub[adata_sub.obs['full_leiden'].isin(keep_groups)].copy()

# 4. Remove 'unused' categories so Scanpy doesn't look for the empty ones
adata_sub.obs['full_leiden'] = adata_sub.obs['full_leiden'].cat.remove_unused_categories()

# 5. Now run your ranking
sc.tl.rank_genes_groups(
    adata_sub, 
    groupby="full_leiden",
)

In [ ]:
df=sc.get.rank_genes_groups_df(adata_sub, group='16')

In [ ]:
# Filter for genes with high fold change and significance
top_markers = df[(df['pvals_adj'] < 0.05) & (df['logfoldchanges'] > 1.5)]

# Sort by the highest logfoldchange
top_markers = top_markers.sort_values('logfoldchanges', ascending=False)

print(top_markers.head(150))

In [ ]:
pd.set_option('display.max_rows', 150)
pd.set_option('display.max_columns', None)  # Ensures all columns like logFC and pval show up
pd.set_option('display.width', 1000)        # Prevents line wrapping

print(top_markers.head(150))

## 4. Composition assessment

In [ ]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# 1. Extract metadata - ensure categories are handled correctly
df = adata_query_X.obs[['individualID', 'comparison_group', 'predictions']].copy()

# 2. Fix the FutureWarning and ensure we have a dense table
# observed=True only looks at categories present in the data
counts = df.groupby(['individualID', 'comparison_group', 'predictions'], observed=True).size().unstack(fill_value=0)

# 3. Convert to proportions
proportions = counts.div(counts.sum(axis=1), axis=0).reset_index()

results = []
cell_types = counts.columns

for cell_type in cell_types:
    # Use .values to ensure we are passing numpy arrays to scipy
    group1 = proportions[proportions['comparison_group'] == 'AD'][cell_type].values
    group2 = proportions[proportions['comparison_group'] == 'PathologyControl'][cell_type].values
    
    # Check if we have enough data points in both groups
    if len(group1) > 0 and len(group2) > 0:
        try:
            stat, p_val = mannwhitneyu(group1, group2, alternative='two-sided')
        except ValueError:
            # This handles cases where all values are identical (e.g., all zeros)
            p_val = 1.0
    else:
        p_val = np.nan

    results.append({
        'Cell Type': cell_type,
        'Mean_AD': np.mean(group1),
        'Mean_Control': np.mean(group2),
        'p-value': p_val
    })

# 5. Build results and calculate Log2FC safely
results_df = pd.DataFrame(results)
results_df['Log2FC'] = np.log2((results_df['Mean_AD'] + 1e-6) / (results_df['Mean_Control'] + 1e-6))

# 6. Apply FDR correction only on non-NaN p-values
mask = ~results_df['p-value'].isna()
results_df.loc[mask, 'FDR_adj_p_val'] = multipletests(results_df.loc[mask, 'p-value'], method='fdr_bh')[1]

print(results_df.sort_values('p-value'))